In [1]:
from google.colab import drive
drive.mount("/content/drive")


Mounted at /content/drive


In [2]:
# =========================================================
# Imports
# =========================================================
from pathlib import Path
from collections import Counter
import ast
import json
import platform
import socket

import pandas as pd

pd.set_option("display.max_columns", 200)
pd.set_option("display.max_colwidth", 300)
pd.set_option("display.width", 200)

print("Libraries loaded.")

Libraries loaded.


In [3]:
# =========================================================
# Path Setup
# =========================================================
NOTEBOOK_NAME = "01_data_check_face.ipynb"
CATEGORY_ID = "face"
CATEGORY_KEY = "facial_skincare"
CATEGORY_FOLDER = "facial_skincare"
CATEGORY_LABEL = "Facial Skincare"
STAGE = "stage0_data_check"

PROJECT_ROOT = Path("/content/drive/MyDrive/thesis_recsys/categories/facial_skincare")
# Raw category-level starting datasets are canonical under data/raw.
# data/derived/pillar1_datasets is legacy and should not be used as the raw input contract.
DATA_DIR = PROJECT_ROOT / "data" / "raw"
SUMMARY_DIR = PROJECT_ROOT / "outputs" / "stage0_data_check"

ITEMS_PATH = PROJECT_ROOT / "data" / "raw" / "items_Skin_Care_Face_W2_2019_2023.parquet"
REVIEWS_PATH = PROJECT_ROOT / "data" / "raw" / "reviews_Skin_Care_Face_W2_2019_2023.parquet"
RESULTS_OVERALL_PATH = SUMMARY_DIR / "results_overall.csv"
DIAGNOSTICS_SUMMARY_PATH = SUMMARY_DIR / "diagnostics_summary.csv"

SUMMARY_DIR.mkdir(parents=True, exist_ok=True)

CONFIG = {
    "notebook_name": NOTEBOOK_NAME,
    "category_id": CATEGORY_ID,
    "category_key": CATEGORY_KEY,
    "category_folder": CATEGORY_FOLDER,
    "category_label": CATEGORY_LABEL,
    "stage": STAGE,
    "project_root": str(PROJECT_ROOT),
    "data_dir": str(DATA_DIR),
    "summary_dir": str(SUMMARY_DIR),
    "items_path": str(ITEMS_PATH),
    "reviews_path": str(REVIEWS_PATH),
}

REQUIRED_INPUT_PATHS = {
    "items_source": ITEMS_PATH,
    "reviews_source": REVIEWS_PATH,
}
missing_inputs = {name: str(path) for name, path in REQUIRED_INPUT_PATHS.items() if not path.exists()}
if missing_inputs:
    raise FileNotFoundError(f"Missing required input files: {missing_inputs}")

print("PROJECT_ROOT:", PROJECT_ROOT)
print("ITEMS_PATH:", ITEMS_PATH)
print("REVIEWS_PATH:", REVIEWS_PATH)
print("SUMMARY_DIR:", SUMMARY_DIR)

PROJECT_ROOT: /content/drive/MyDrive/thesis_recsys/categories/facial_skincare
ITEMS_PATH: /content/drive/MyDrive/thesis_recsys/categories/facial_skincare/data/raw/items_Skin_Care_Face_W2_2019_2023.parquet
REVIEWS_PATH: /content/drive/MyDrive/thesis_recsys/categories/facial_skincare/data/raw/reviews_Skin_Care_Face_W2_2019_2023.parquet
SUMMARY_DIR: /content/drive/MyDrive/thesis_recsys/categories/facial_skincare/outputs/stage0_data_check


In [4]:
# =========================================================
# Data Loading
# =========================================================
items = pd.read_parquet(ITEMS_PATH)
reviews = pd.read_parquet(REVIEWS_PATH)

print("items shape :", items.shape)
print("reviews shape:", reviews.shape)

items shape : (78219, 12)
reviews shape: (990619, 9)


In [5]:
# =========================================================
# Schema Inspection
# =========================================================
print("Items columns:")
print(items.columns.tolist())
print()

print("Reviews columns:")
print(reviews.columns.tolist())
print()

print("Items dtypes:")
print(items.dtypes)
print()

print("Reviews dtypes:")
print(reviews.dtypes)

Items columns:
['main_category', 'title', 'average_rating', 'rating_number', 'features', 'description', 'price', 'brand', 'categories', 'details', 'parent_asin', 'bought_together']

Reviews columns:
['rating', 'title', 'text', 'asin', 'parent_asin', 'user_id', 'timestamp', 'helpful_vote', 'verified_purchase']

Items dtypes:
main_category      object
title              object
average_rating     object
rating_number      object
features           object
description        object
price              object
brand              object
categories         object
details            object
parent_asin        object
bought_together    object
dtype: object

Reviews dtypes:
rating               object
title                object
text                 object
asin                 object
parent_asin          object
user_id              object
timestamp            object
helpful_vote         object
verified_purchase    object
dtype: object


In [6]:
# =========================================================
# Data Preview
# =========================================================
print("Items head:")
display(items.head(5))

print("Reviews head:")
display(reviews.head(5))

Items head:


,main_category,title,average_rating,rating_number,features,description,price,brand,categories,details,parent_asin,bought_together
0,All Beauty,Gone in Sixty Seconds Instant Wrinkle Eraser,3.5,98,"[""New more powerful instant lifting and tightening effect with patented Dynalift"", ""Instant lift and temporary wrinkle eraser with long term anti wrinkle benefits"", ""Fill in crows feet and deep creases"", ""Smooth the look of puffiness and bags around the eye area"", ""Leaves no residue and works on...","[""What it is: Give us 15 seconds and we will give you 15 years. Gone In 60 Seconds is a powerful formula that activates on contact to visibly erase fine lines and deep wrinkles in just seconds.""]",20.89,AminoGenesis,"[""Beauty & Personal Care"", ""Skin Care"", ""Face"", ""Treatments & Masks""]","{""Is Discontinued By Manufacturer"": ""No"", ""Package Dimensions"": ""5.2 x 1.34 x 1.26 inches; 1.13 Ounces"", ""UPC"": ""858322000358""}",B01KQNJCNG,None
1,All Beauty,"Asara New York Probiotic Skincare Rejuvenating Toner, 30 ml (1.01 fl oz)",5.0,5,"[""Use to provide extra, on-demand hydration and protection to the skin throughout the day with a simple spritz from the pump"", ""Packed with a powerful blend of probiotics researched for their specific effects on skin health and appearance"", ""Supercharged with organic antioxidants including sea b...",[],None,ASARA NEW YORK PROBIOTIC SKIN SCIENCE,"[""Beauty & Personal Care"", ""Skin Care"", ""Face"", ""Toners & Astringents""]","{""Brand"": ""ASARA NEW YORK PROBIOTIC SKIN SCIENCE"", ""Skin Type"": ""Oily, Combination, Dry, Normal"", ""Material Feature"": ""Organic, Natural"", ""Item Volume"": ""30 Milliliters"", ""Active Ingredients"": ""hyaluronic_acid"", ""Is Discontinued By Manufacturer"": ""No"", ""Package Dimensions"": ""5.5 x 1.25 x 1.25 in...",B07FXWHFRF,None
2,All Beauty,Skinfix Resurface AHA Renewing Cream 0.35 fl.oz. 10 ml. Travel size New,4.2,15,"[""Skinfix Resurface AHA Renewing Cream 0.35 fl.oz. 10 ml. Travel size New"", ""Formulation: Cream"", ""Brand: Skinfix"", ""Type: Day Cream"", ""Size: Travel Size""]","[""Condition: New without box: A brand-new, unused, and unworn item (including handmade items) that is not in original packaging or may be missing original packaging materials (such as the original box or bag). The original tags may not be attached. For example, new shoes (with absolutely no sign...",None,Bissport,"[""Beauty & Personal Care"", ""Skin Care"", ""Face"", ""Creams & Moisturizers"", ""Face Moisturizers""]","{""Brand"": ""Bissport"", ""Item Form"": ""Cream"", ""Unit Count"": ""1.00 Count"", ""Number of Items"": ""1"", ""Use for"": ""Face"", ""Manufacturer"": ""Bissport""}",B08PFNJF1X,None
3,All Beauty,"Holika Holika Pig Nose Clear Black Head Cleansing Sugar Scrub, 1.01 Ounce",4.2,17,"[""It contains soft white sugar for exfoliating"", ""It removes dead skin cells"", ""For easier extraction of blackheads""]","[""This is a wash-off scrub to exfoliate blackhead. It contains soft white sugar for exfoliating. It removes dead skin cells and allowing for easier extraction of blackheads and other impurities from pores. Pink clay ingredients help to absorb excess sebum. Lemon extracts and Aloe Vera extracts p...",7.99,HOLIKA HOLIKA,"[""Beauty & Personal Care"", ""Skin Care"", ""Face"", ""Cleansers"", ""Gels""]","{""Brand"": ""HOLIKA HOLIKA"", ""Skin Type"": ""Acne prone"", ""Product Benefits"": ""Exfoliate,Cleansing,Exfoliating"", ""Item Volume"": ""25 Milliliters"", ""Number of Items"": ""1"", ""Is Discontinued By Manufacturer"": ""No"", ""Product Dimensions"": ""1.5 x 1.5 x 1.5 inches; 0.35 Ounces"", ""Item model number"": ""200117...",B00EBZEAH4,None
4,All Beauty,"Alba Botanica Hemp Soothing Serum, 1 oz",4.2,30,"[""One 1 oz. bottle of Alba Botanica Hemp Soothing Serum"", ""Helps soothe skin and boost hydration for a brighter overall complexion"", ""Made with a plant-based blend of botanical ingredients for sensitive skin"", ""100% vegetarian formula made without har

Reviews head:


,rating,title,text,asin,parent_asin,user_id,timestamp,helpful_vote,verified_purchase
0,5.0,What a nice scrub this is.,Smells so good but light too. The scrub is effective and the product rinses well. You will need moisturizer afterward. I live in AZ so I'm always dry.,B07L1DVWT9,B0B77ZL2YP,AGKHLEW2SOWHNMFQIJGBECAF7INQ,1588687504456,0,True
1,2.0,Overpriced.,Does not help my neck.,B00Q6SALWC,B0BGNLBF8W,AHITBJSS7KYUBVZPX7M2WJCOIVKQ,1630165089593,0,True
2,5.0,Great cleanser!,Removes makeup and dirt like magic.,B01MQ495N0,B09JN3S1X9,AHITBJSS7KYUBVZPX7M2WJCOIVKQ,1560014237021,0,True
3,5.0,Clinique All About Clean 2-in-1 Cleansing exfoliator,"Clinique All About Clean 2-in-1 Cleansing Exfoliator. It makes your face feel extra clean and smooth. The exfoliators are wonderful at refreshing dry, dull skin",B094RDG1JH,B0B8QXHJX4,AHOEIYJJHZ7ITX75BOFQYNXVVJQQ,1671288550526,1,True
4,4.0,Handy for travelingp,It’s good but there isn’t very much of them for the price,B078ZFYRF9,B078ZFYRF9,AFNT6ZJCYQN3WDIKUSWHJDXNND2Q,1573343933031,0,True


In [7]:
# =========================================================
# Null Summary
# =========================================================
items_nulls = pd.DataFrame({
    "null_count": items.isna().sum(),
    "null_ratio": items.isna().mean().round(4)
}).sort_values(["null_ratio", "null_count"], ascending=False)

reviews_nulls = pd.DataFrame({
    "null_count": reviews.isna().sum(),
    "null_ratio": reviews.isna().mean().round(4)
}).sort_values(["null_ratio", "null_count"], ascending=False)

print("Items null summary:")
display(items_nulls)

print("Reviews null summary:")
display(reviews_nulls)

Items null summary:


,null_count,null_ratio
bought_together,78219,1.0000
price,44416,0.5678
brand,7123,0.0911
main_category,6043,0.0773
title,0,0.0000
average_rating,0,0.0000
rating_number,0,0.0000
features,0,0.0000
description,0,0.0000
categories,0,0.0000


Reviews null summary:


,null_count,null_ratio
rating,0,0.0
title,0,0.0
text,0,0.0
asin,0,0.0
parent_asin,0,0.0
user_id,0,0.0
timestamp,0,0.0
helpful_vote,0,0.0
verified_purchase,0,0.0


In [8]:
# =========================================================
# Cardinality Check
# =========================================================
def safe_nunique(series):
    try:
        return series.nunique(dropna=True)
    except Exception:
        return None

items_cardinality = pd.DataFrame({
    "dtype": items.dtypes.astype(str),
    "nunique": [safe_nunique(items[c]) for c in items.columns]
}).sort_values("nunique", ascending=False)

reviews_cardinality = pd.DataFrame({
    "dtype": reviews.dtypes.astype(str),
    "nunique": [safe_nunique(reviews[c]) for c in reviews.columns]
}).sort_values("nunique", ascending=False)

print("Items cardinality:")
display(items_cardinality)

print("Reviews cardinality:")
display(reviews_cardinality)

Items cardinality:


,dtype,nunique
parent_asin,object,78219
details,object,75164
title,object,74350
features,object,51367
description,object,36619
brand,object,17962
price,object,4612
rating_number,object,3351
average_rating,object,41
main_category,object,32


Reviews cardinality:


,dtype,nunique
timestamp,object,981068
text,object,919712
user_id,object,747912
title,object,521476
asin,object,59378
parent_asin,object,51420
helpful_vote,object,432
rating,object,5
verified_purchase,object,2


In [9]:
# =========================================================
# Text Length Summary
# =========================================================
def text_len_stats(df, col):
    s = df[col].fillna("").astype(str)
    return pd.Series({
        "non_null": df[col].notna().sum(),
        "avg_len": round(s.str.len().mean(), 2),
        "median_len": round(s.str.len().median(), 2),
        "max_len": s.str.len().max()
    })

length_stats = {}

for col in ["title", "features", "description", "details", "categories", "brand"]:
    if col in items.columns:
        length_stats[f"items::{col}"] = text_len_stats(items, col)

for col in ["title", "text"]:
    if col in reviews.columns:
        length_stats[f"reviews::{col}"] = text_len_stats(reviews, col)

length_stats_df = pd.DataFrame(length_stats).T
display(length_stats_df)

,non_null,avg_len,median_len,max_len
items::title,78219.0,103.49,86.0,977.0
items::features,78219.0,471.25,239.0,4367.0
items::description,78219.0,316.97,44.0,19014.0
items::details,78219.0,267.19,263.0,2412.0
items::categories,78219.0,77.99,78.0,94.0
items::brand,71096.0,9.36,8.0,156.0
reviews::title,990619.0,21.47,18.0,101.0
reviews::text,990619.0,212.44,129.0,17174.0


In [10]:
# =========================================================
# Details Structure
# =========================================================
def parse_maybe_dict(x):
    if pd.isna(x):
        return None
    if isinstance(x, dict):
        return x
    if isinstance(x, str):
        x = x.strip()
        if not x:
            return None
        try:
            return ast.literal_eval(x)
        except Exception:
            try:
                return json.loads(x)
            except Exception:
                return None
    return None

details_parsed = items["details"].apply(parse_maybe_dict) if "details" in items.columns else pd.Series(dtype="object")

print("Parsed details non-null count:", details_parsed.notna().sum())

sample_details = details_parsed.dropna().head(10).tolist()
for i, d in enumerate(sample_details, start=1):
    print(f"\nSample details {i}:")
    print(type(d))
    print(list(d.items())[:10] if isinstance(d, dict) else d)

Parsed details non-null count: 78219

Sample details 1:
<class 'dict'>
[('Is Discontinued By Manufacturer', 'No'), ('Package Dimensions', '5.2 x 1.34 x 1.26 inches; 1.13 Ounces'), ('UPC', '858322000358')]

Sample details 2:
<class 'dict'>
[('Brand', 'ASARA NEW YORK PROBIOTIC SKIN SCIENCE'), ('Skin Type', 'Oily, Combination, Dry, Normal'), ('Material Feature', 'Organic, Natural'), ('Item Volume', '30 Milliliters'), ('Active Ingredients', 'hyaluronic_acid'), ('Is Discontinued By Manufacturer', 'No'), ('Package Dimensions', '5.5 x 1.25 x 1.25 inches; 3.21 Ounces'), ('UPC', '816554010859'), ('Manufacturer', 'Asara New York')]

Sample details 3:
<class 'dict'>
[('Brand', 'Bissport'), ('Item Form', 'Cream'), ('Unit Count', '1.00 Count'), ('Number of Items', '1'), ('Use for', 'Face'), ('Manufacturer', 'Bissport')]

Sample details 4:
<class 'dict'>
[('Brand', 'HOLIKA HOLIKA'), ('Skin Type', 'Acne prone'), ('Product Benefits', 'Exfoliate,Cleansing,Exfoliating'), ('Item Volume', '25 Milliliters'

In [11]:
# =========================================================
# Details Key Frequency
# =========================================================
from collections import Counter

detail_key_counter = Counter()

for d in details_parsed.dropna():
    if isinstance(d, dict):
        detail_key_counter.update(d.keys())

detail_key_df = pd.DataFrame(
    detail_key_counter.most_common(100),
    columns=["detail_key", "count"]
)

print("Top detail keys:")
display(detail_key_df)

Top detail keys:


,detail_key,count
0,Manufacturer,61546
1,Brand,59757
2,Item Form,46868
3,UPC,43743
4,Is Discontinued By Manufacturer,37084
...,...,...
95,Is Dishwasher Safe,33
96,Assembly required,32
97,Compatible Phone Models,31
98,Whats in the box,31


In [12]:
# =========================================================
# Feature Structure
# =========================================================
for col in ["features", "description"]:
    if col in items.columns:
        print(f"\nColumn structure check: {col}")
        sample_vals = items[col].dropna().head(10).tolist()
        for i, v in enumerate(sample_vals, start=1):
            print(f"\nSample {i} type={type(v)}")
            print(str(v)[:500])


Column structure check: features

Sample 1 type=<class 'str'>
["New more powerful instant lifting and tightening effect with patented Dynalift", "Instant lift and temporary wrinkle eraser with long term anti wrinkle benefits", "Fill in crows feet and deep creases", "Smooth the look of puffiness and bags around the eye area", "Leaves no residue and works on all skin types Results last all day until washed off"]

Sample 2 type=<class 'str'>
["Use to provide extra, on-demand hydration and protection to the skin throughout the day with a simple spritz from the pump", "Packed with a powerful blend of probiotics researched for their specific effects on skin health and appearance", "Supercharged with organic antioxidants including sea buckthorn, wild chamomile, juniper berry, lingonberry, blackcurrant, and argan to help moisturize and protect skin", "Features a perfectly balanced fragrance, making it a customer favorite; Designed to all

Sample 3 type=<class 'str'>
["Skinfix Resurface AHA Re

In [13]:
# =========================================================
# Category Structure
# =========================================================
for col in ["categories", "bought_together"]:
    if col in items.columns:
        print(f"\nColumn structure check: {col}")
        sample_vals = items[col].dropna().head(10).tolist()
        for i, v in enumerate(sample_vals, start=1):
            print(f"\nSample {i} type={type(v)}")
            print(str(v)[:500])


Column structure check: categories

Sample 1 type=<class 'str'>
["Beauty & Personal Care", "Skin Care", "Face", "Treatments & Masks"]

Sample 2 type=<class 'str'>
["Beauty & Personal Care", "Skin Care", "Face", "Toners & Astringents"]

Sample 3 type=<class 'str'>
["Beauty & Personal Care", "Skin Care", "Face", "Creams & Moisturizers", "Face Moisturizers"]

Sample 4 type=<class 'str'>
["Beauty & Personal Care", "Skin Care", "Face", "Cleansers", "Gels"]

Sample 5 type=<class 'str'>
["Beauty & Personal Care", "Skin Care", "Face", "Treatments & Masks", "Serums"]

Sample 6 type=<class 'str'>
["Beauty & Personal Care", "Skin Care", "Face", "Creams & Moisturizers", "Face Moisturizers"]

Sample 7 type=<class 'str'>
["Beauty & Personal Care", "Skin Care", "Face", "Creams & Moisturizers", "Face Moisturizers"]

Sample 8 type=<class 'str'>
["Beauty & Personal Care", "Skin Care", "Face"]

Sample 9 type=<class 'str'>
["Beauty & Personal Care", "Skin Care", "Face", "Cleansers", "Washes"]

Sample 10 

In [14]:
# =========================================================
# Item Review Distribution
# =========================================================
if "parent_asin" in reviews.columns:
    review_per_item = reviews.groupby("parent_asin").size().rename("n_reviews").reset_index()

    print("Review count per parent_asin summary:")
    print(review_per_item["n_reviews"].describe())

    print("\nTop reviewed items:")
    display(review_per_item.sort_values("n_reviews", ascending=False).head(20))

    print("\nBottom reviewed items:")
    display(review_per_item.sort_values("n_reviews", ascending=True).head(20))

Review count per parent_asin summary:
count    51420.000000
mean        19.265247
std        128.140065
min          1.000000
25%          1.000000
50%          3.000000
75%         10.000000
max      14741.000000
Name: n_reviews, dtype: float64

Top reviewed items:


,parent_asin,n_reviews
48966,B0BS71PXPX,14741
50335,B0C37PFCWW,9251
33316,B08MVQ1X2P,6787
49505,B0BVW6XYNZ,5875
7752,B00NR1YQHM,4956
47034,B0BG9Q18ZZ,4066
33329,B08MWTFNNK,3951
8518,B00U2VQZDS,3595
23797,B07QXBS32K,3475
50456,B0C46SPKJ9,3400



Bottom reviewed items:


,parent_asin,n_reviews
37452,B094QKMJLQ,1
10,5455896636,1
34,B0000530NA,1
38,B0000534VO,1
40,B0000535OF,1
42,B0000535UZ,1
51393,B0CHN9929P,1
51396,B0CHQL3163,1
15,B000052YJN,1
16,B000052YJU,1


In [15]:
# =========================================================
# User Review Distribution
# =========================================================
if "user_id" in reviews.columns:
    review_per_user = reviews.groupby("user_id").size().rename("n_reviews").reset_index()

    print("Review count per user summary:")
    print(review_per_user["n_reviews"].describe())

    repeat_bucket = pd.cut(
        review_per_user["n_reviews"],
        bins=[0, 1, 4, 9, 19, review_per_user["n_reviews"].max()],
        labels=["1", "2-4", "5-9", "10-19", "20+"],
        include_lowest=True
    )

    user_bucket_summary = repeat_bucket.value_counts(dropna=False).sort_index().reset_index()
    user_bucket_summary.columns = ["repeat_bucket", "n_users"]
    user_bucket_summary["user_ratio"] = user_bucket_summary["n_users"] / user_bucket_summary["n_users"].sum()

    print("\nUser repeat bucket summary:")
    display(user_bucket_summary)

Review count per user summary:
count    747912.000000
mean          1.324513
std           3.079425
min           1.000000
25%           1.000000
50%           1.000000
75%           1.000000
max         755.000000
Name: n_reviews, dtype: float64

User repeat bucket summary:


,repeat_bucket,n_users,user_ratio
0,1,635096,0.849159
1,2-4,103729,0.138691
2,5-9,6522,0.008720
3,10-19,1500,0.002006
4,20+,1065,0.001424


In [16]:
# =========================================================
# Rating Timestamp Check
# =========================================================
if "rating" in reviews.columns:
    print("Rating distribution:")
    display(reviews["rating"].value_counts(dropna=False).sort_index())

if "verified_purchase" in reviews.columns:
    print("Verified purchase distribution:")
    display(reviews["verified_purchase"].value_counts(dropna=False))

if "helpful_vote" in reviews.columns:
    print("Helpful vote summary:")
    print(pd.to_numeric(reviews["helpful_vote"], errors="coerce").describe())

if "timestamp" in reviews.columns:
    ts = pd.to_datetime(pd.to_numeric(reviews["timestamp"], errors="coerce"), unit="ms", errors="coerce")
    print("Timestamp min:", ts.min())
    print("Timestamp max:", ts.max())
    print("Review year distribution:")
    display(ts.dt.year.value_counts().sort_index())

Rating distribution:


,count
rating,
1.0,109779
2.0,44805
3.0,62903
4.0,101805
5.0,671327


Verified purchase distribution:


,count
verified_purchase,
True,863217
False,127402


Helpful vote summary:
count    990619.000000
mean          1.260131
std          12.188462
min           0.000000
25%           0.000000
50%           0.000000
75%           1.000000
max        3414.000000
Name: helpful_vote, dtype: float64
Timestamp min: 2019-01-01 00:01:33.712000
Timestamp max: 2023-09-12 14:52:26.427000
Review year distribution:


,count
timestamp,
2019,185909
2020,227293
2021,232679
2022,236769
2023,107969


In [17]:
# =========================================================
# Brand Check
# =========================================================
if "brand" in items.columns:
    brand_counts = (
        items["brand"]
        .fillna("")
        .astype(str)
        .str.strip()
        .value_counts()
        .reset_index()
    )
    brand_counts.columns = ["brand", "n_items"]

    print("Top brands by item count:")
    display(brand_counts.head(30))

Top brands by item count:


,brand,n_items
0,,7123
1,Olay,751
2,Neutrogena,666
3,Clinique,626
4,Generic,313
5,Estee Lauder,302
6,Origins,266
7,Eminence,255
8,Clean & Clear,255
9,Mary Kay,253


In [18]:
# =========================================================
# Output Export
# =========================================================
output_dir = SUMMARY_DIR
created_outputs = []

base_outputs = {
    "items_null_summary.csv": items_nulls,
    "reviews_null_summary.csv": reviews_nulls,
    "items_cardinality.csv": items_cardinality,
    "reviews_cardinality.csv": reviews_cardinality,
    "text_length_stats.csv": length_stats_df,
    "detail_key_frequency.csv": detail_key_df,
    "items_preview.csv": items.head(100),
    "reviews_preview.csv": reviews.head(100),
}

for filename, df in base_outputs.items():
    path = output_dir / filename
    df.to_csv(path, index=True if filename.endswith(("null_summary.csv", "cardinality.csv", "text_length_stats.csv")) else False, encoding="utf-8-sig")
    created_outputs.append(path)

optional_outputs = {
    "review_per_item.csv": globals().get("review_per_item"),
    "review_per_user.csv": globals().get("review_per_user"),
    "user_repeat_bucket_summary.csv": globals().get("user_bucket_summary"),
    "brand_item_counts.csv": globals().get("brand_counts"),
}

for filename, df in optional_outputs.items():
    if df is not None:
        path = output_dir / filename
        df.to_csv(path, index=False, encoding="utf-8-sig")
        created_outputs.append(path)

results_overall = pd.DataFrame([{
    "notebook_name": NOTEBOOK_NAME,
    "category_id": CATEGORY_ID,
    "category_key": CATEGORY_KEY,
    "category_folder": CATEGORY_FOLDER,
    "category_label": CATEGORY_LABEL,
    "stage": STAGE,
    "input_items_path": str(ITEMS_PATH),
    "input_reviews_path": str(REVIEWS_PATH),
    "output_root": str(output_dir),
    "n_rows_items": len(items),
    "n_rows_reviews": len(reviews),
    "n_cols_items": items.shape[1],
    "n_cols_reviews": reviews.shape[1],
    "status": "completed",
}])
results_overall.to_csv(RESULTS_OVERALL_PATH, index=False, encoding="utf-8-sig")
created_outputs.append(RESULTS_OVERALL_PATH)

diagnostics_summary = pd.DataFrame([
    {
        "notebook_name": NOTEBOOK_NAME,
        "category_id": CATEGORY_ID,
        "category_key": CATEGORY_KEY,
        "category_folder": CATEGORY_FOLDER,
        "category_label": CATEGORY_LABEL,
        "stage": STAGE,
        "check_name": "items_input_exists",
        "path": str(ITEMS_PATH),
        "check_passed": ITEMS_PATH.exists(),
        "row_count": len(items),
        "note": "curated items input",
    },
    {
        "notebook_name": NOTEBOOK_NAME,
        "category_id": CATEGORY_ID,
        "category_key": CATEGORY_KEY,
        "category_folder": CATEGORY_FOLDER,
        "category_label": CATEGORY_LABEL,
        "stage": STAGE,
        "check_name": "reviews_input_exists",
        "path": str(REVIEWS_PATH),
        "check_passed": REVIEWS_PATH.exists(),
        "row_count": len(reviews),
        "note": "curated reviews input",
    },
    {
        "notebook_name": NOTEBOOK_NAME,
        "category_id": CATEGORY_ID,
        "category_key": CATEGORY_KEY,
        "category_folder": CATEGORY_FOLDER,
        "category_label": CATEGORY_LABEL,
        "stage": STAGE,
        "check_name": "items_has_rows",
        "path": str(ITEMS_PATH),
        "check_passed": len(items) > 0,
        "row_count": len(items),
        "note": "basic row-count check",
    },
    {
        "notebook_name": NOTEBOOK_NAME,
        "category_id": CATEGORY_ID,
        "category_key": CATEGORY_KEY,
        "category_folder": CATEGORY_FOLDER,
        "category_label": CATEGORY_LABEL,
        "stage": STAGE,
        "check_name": "reviews_has_rows",
        "path": str(REVIEWS_PATH),
        "check_passed": len(reviews) > 0,
        "row_count": len(reviews),
        "note": "basic row-count check",
    },
])
diagnostics_summary.to_csv(DIAGNOSTICS_SUMMARY_PATH, index=False, encoding="utf-8-sig")
created_outputs.append(DIAGNOSTICS_SUMMARY_PATH)

print("Data check notebook completed.")
print("Items rows:", len(items))
print("Reviews rows:", len(reviews))
print("Summary directory:", SUMMARY_DIR)
print("Results summary:", RESULTS_OVERALL_PATH)
print("Diagnostics summary:", DIAGNOSTICS_SUMMARY_PATH)

display(results_overall)
display(diagnostics_summary)

Data check notebook completed.
Items rows: 78219
Reviews rows: 990619
Summary directory: /content/drive/MyDrive/thesis_recsys/categories/facial_skincare/outputs/stage0_data_check
Results summary: /content/drive/MyDrive/thesis_recsys/categories/facial_skincare/outputs/stage0_data_check/results_overall.csv
Diagnostics summary: /content/drive/MyDrive/thesis_recsys/categories/facial_skincare/outputs/stage0_data_check/diagnostics_summary.csv


,notebook_name,category_id,category_key,category_folder,category_label,stage,input_items_path,input_reviews_path,output_root,n_rows_items,n_rows_reviews,n_cols_items,n_cols_reviews,status
0,01_data_check_face.ipynb,face,facial_skincare,facial_skincare,Facial Skincare,stage0_data_check,/content/drive/MyDrive/thesis_recsys/categories/facial_skincare/data/raw/items_Skin_Care_Face_W2_2019_2023.parquet,/content/drive/MyDrive/thesis_recsys/categories/facial_skincare/data/raw/reviews_Skin_Care_Face_W2_2019_2023.parquet,/content/drive/MyDrive/thesis_recsys/categories/facial_skincare/outputs/stage0_data_check,78219,990619,12,9,completed


,notebook_name,category_id,category_key,category_folder,category_label,stage,check_name,path,check_passed,row_count,note
0,01_data_check_face.ipynb,face,facial_skincare,facial_skincare,Facial Skincare,stage0_data_check,items_input_exists,/content/drive/MyDrive/thesis_recsys/categories/facial_skincare/data/raw/items_Skin_Care_Face_W2_2019_2023.parquet,True,78219,curated items input
1,01_data_check_face.ipynb,face,facial_skincare,facial_skincare,Facial Skincare,stage0_data_check,reviews_input_exists,/content/drive/MyDrive/thesis_recsys/categories/facial_skincare/data/raw/reviews_Skin_Care_Face_W2_2019_2023.parquet,True,990619,curated reviews input
2,01_data_check_face.ipynb,face,facial_skincare,facial_skincare,Facial Skincare,stage0_data_check,items_has_rows,/content/drive/MyDrive/thesis_recsys/categories/facial_skincare/data/raw/items_Skin_Care_Face_W2_2019_2023.parquet,True,78219,basic row-count check
3,01_data_check_face.ipynb,face,facial_skincare,facial_skincare,Facial Skincare,stage0_data_check,reviews_has_rows,/content/drive/MyDrive/thesis_recsys/categories/facial_skincare/data/raw/reviews_Skin_Care_Face_W2_2019_2023.parquet,True,990619,basic row-count check
